# soft_prompt_val_gen_ii — variant (ii), val slice at four checkpoints
Variant (ii): P learnable rows appended to the MLLM **output**, i.e. straight onto the conditioning stream the diffusion transformer attends to. The rows never pass through the MLLM, so they add conditioning rather than reinterpreting the instruction. Generates the 98-row val slice at steps 500 / 1000 / 2000 / 3000 for checkpoint selection.

Sampler, negative prompt, reference handling and instruction handling are identical to every other arm; the learned prompt is the only difference. The base weights are untouched, so one pipeline is built and the prompt tensor is swapped per checkpoint — no per-step rebuild.

Output: `gen/val/sp_ii_p668__step{N}/{i:05d}.png`. Resumable per step. Scored afterwards into `scores/val__sp_ii_p668_step{N}.jsonl`.

Cost: 4 x 98 rows x 23.5 s = 2.6 h A100.

## 1. Repo

In [ ]:
import os, subprocess, sys
REPO = "/content/OmniGen2"
COMMIT = "18e6f9d5271b517fcb32e999f10df943ae9b8f20"   # the tree every arm of this thesis ran on
if not os.path.isdir(REPO):
    subprocess.run(["git","clone","https://github.com/VectorSpaceLab/OmniGen2.git",REPO], check=True)
    subprocess.run(["git","-C",REPO,"checkout","--quiet",COMMIT], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"], cwd=REPO, check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","-U","huggingface_hub[hf_xet]>=0.30.0,<1.0"], check=True)
    print("installed.")
else:
    print("OmniGen2 already present.")

_head = subprocess.run(["git","-C",REPO,"rev-parse","HEAD"],
                       capture_output=True, text=True).stdout.strip()
print(_head[:7], "= repo commit")
# Everything this notebook asserts about encode_prompt, the three CFG branches and the rope tables
# was read off this tree. A moved upstream HEAD would change them while every check below still
# passed, so the commit is pinned rather than reported.
assert _head == COMMIT, (
    f"repo is at {_head[:7]}, expected {COMMIT[:7]}. Run: git -C {REPO} checkout {COMMIT}")
sys.path.insert(0, REPO)
os.chdir(REPO)

installed.
18e6f9d = repo commit


## 2. Drive, run identity, prompt checkpoints

In [ ]:
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DRIVE   = "/content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data"
DATASET = "val"                 # 98-row held-out slice, the checkpoint-selection set
VARIANT = "ii"
RUN     = "sp_ii_p668"
P       = 668
KEY     = "soft_prompt"               # tensor key inside prompt-<step>.pt
STEPS   = [500, 1000, 2000, 3000]   # the four scored checkpoints; 1500 and 2500 stay unscored on disk
EXP     = f"{DRIVE}/soft_prompt/experiments/{RUN}"

def ckpt_path(step):
    return f"{EXP}/prompt-{step}.pt"

for step in STEPS:
    assert os.path.exists(ckpt_path(step)), f"checkpoint missing: {ckpt_path(step)}"
print(f"{RUN}: all {len(STEPS)} scored checkpoints present in {EXP}")

Mounted at /content/drive
sp_ii_p668: all 4 scored checkpoints present in /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/soft_prompt/experiments/sp_ii_p668


## 3. Images, split

In [ ]:
import subprocess, json
IMGS, EXPECTED = "/content/imgs", 37063
TAR = f"{DRIVE}/imgs_subset.tar"

def n_files(d):
    return sum(len(f) for _, _, f in os.walk(d)) if os.path.isdir(d) else 0

if n_files(IMGS) < EXPECTED:
    os.makedirs(IMGS, exist_ok=True)
    subprocess.run(f'cp "{TAR}" /content/imgs_subset.tar && tar -xf /content/imgs_subset.tar -C {IMGS}',
                   shell=True, check=True)
    print("restored", n_files(IMGS), "/", EXPECTED)
    assert n_files(IMGS) >= EXPECTED, "restore incomplete"
else:
    print("images complete:", n_files(IMGS))

for s in ["train", "val", "test"]:
    src = f"{DRIVE}/{s}.jsonl"
    if os.path.exists(src) and not os.path.exists(f"/content/{s}.jsonl"):
        subprocess.run(["cp", src, f"/content/{s}.jsonl"])

rows = [json.loads(l) for l in open(f"/content/{DATASET}.jsonl", encoding="utf-8")]
missing = [p for r in rows for p in r["input_images"] if not os.path.exists(os.path.join(IMGS, p))]
assert not missing, f"{len(missing)} reference images missing, first: {missing[:3]}"
print(f"{DATASET}: {len(rows)} rows, all reference images present")

restored 37063 / 37063
val: 98 rows, all reference images present


## 4. Engine

In [ ]:
from PIL import Image, ImageOps
import torch

# shipped default from inference.py; quality filter, not a domain negative
NEG = ("(((deformed))), blurry, over saturation, bad anatomy, disfigured, poorly drawn face, mutation, mutated, "
       "(extra_limb), (ugly), (poorly drawn hands), fused fingers, messy drawing, broken legs censor, censored, censor_bar")

MAX_SEQ = 1024

def load_inputs(paths, max_side=512):   # refs downscaled; output size fixed at 512 via align_res=False
    out = []
    for p in paths:
        im = ImageOps.exif_transpose(Image.open(p).convert("RGB"))
        w, h = im.size
        s = max_side / max(w, h)
        if s < 1: w, h = int(w*s), int(h*s)
        w, h = max(16, (w//16)*16), max(16, (h//16)*16)
        out.append(im.resize((w, h), Image.LANCZOS))
    return out

## 5. Pipeline and rope tables
Base weights, no adapter. The rope text axis is extended before any generation; the extension cannot change an existing row.

In [ ]:
import torch, gc
from accelerate import Accelerator
from omnigen2.pipelines.omnigen2.pipeline_omnigen2 import OmniGen2Pipeline
from omnigen2.models.transformers.transformer_omnigen2 import OmniGen2Transformer2DModel

MODEL = "OmniGen2/OmniGen2"
accelerator = Accelerator(mixed_precision="bf16")
dtype = torch.bfloat16
DEV = accelerator.device

# base pipeline, unmodified weights: this arm trains no model parameter, only the prompt tensor
pipe = OmniGen2Pipeline.from_pretrained(MODEL, torch_dtype=dtype, trust_remote_code=True)
pipe.transformer = OmniGen2Transformer2DModel.from_pretrained(MODEL, subfolder="transformer", torch_dtype=dtype)
pipe = pipe.to(DEV)
print("pipeline ready on", DEV)

# The released transformer config caps the text rope axis at 1024. cap_seq_len for the positive
# branch is text_len + P, which exceeds it for large P. get_1d_rotary_pos_embed is
# outer(arange(e), freqs), so row k is identical for any table length above k: enlarging the table
# extends it rather than changing any existing row. Applied unconditionally so the same code holds
# at P = 1,247, where every row overflows.
ROPE_LEN = 4096
_axes_before = tuple(pipe.transformer.config.axes_lens)
pipe.transformer.config.axes_lens = [max(int(l), ROPE_LEN) for l in _axes_before]
print(f"rope axes_lens {_axes_before} -> {tuple(pipe.transformer.config.axes_lens)}")

TOK = pipe.processor.tokenizer

def tokenize(prompt_text):
    """Chat-template then tokenize exactly as _get_qwen2_prompt_embeds does."""
    templated = pipe._apply_chat_template(prompt_text)
    ti = TOK([templated], padding="longest", max_length=MAX_SEQ, truncation=True, return_tensors="pt")
    return ti.input_ids.to(DEV), ti.attention_mask.to(DEV)

def _read(out):
    """train.py read .last_hidden_state off Qwen2_5_VLModel; the pipeline reads .hidden_states[-1]
    off Qwen2_5_VLForConditionalGeneration. Reads whichever the given class exposes so the two can
    be compared directly."""
    lhs = getattr(out, "last_hidden_state", None)
    return lhs if lhs is not None else out.hidden_states[-1]

@torch.no_grad()
def encode_stock(prompt_text, mllm=None):
    """The pipeline's own text path, with no prompt injected."""
    m = mllm if mllm is not None else pipe.mllm
    ids, mask = tokenize(prompt_text)
    feats = _read(m(input_ids=ids, attention_mask=mask, output_hidden_states=True))
    return feats.to(dtype=pipe.mllm.dtype), mask

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/content/OmniGen2/omnigen2/models/attention_processor.py:33: UserWarning: Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use Flash2Varlen attention for better performance")
/content/OmniGen2/omnigen2/models/transformers/block_lumina2.py:37: UserWarning: Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance
  warnings.warn("Cannot import flash_attn, install flash_attn to use fused SwiGLU for better performance")


model_index.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Expected types for transformer: (<class 'omnigen2.models.transformers.transformer_omnigen2.OmniGen2Transformer2DModel'>,), got <class 'diffusers_modules.local.transformer_omnigen2.OmniGen2Transformer2DModel'>.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

pipeline ready on cuda
rope axes_lens (1024, 1664, 1664) -> (4096, 4096, 4096)


## 6. Prompt injection and generation call
The injection mirrors the training patch line for line. CFG is positive-branch only: the learned rows enter through `g_txt * (pos - ref)` and are amplified 5x exactly as the instruction text is in every other arm; putting them in the negative branch would invert their meaning.

In [ ]:
@torch.no_grad()
def encode_injected(prompt_text, mllm=None):
    """Variant (ii): run the MLLM unchanged, then append the learned rows to its output.
    Mirrors the training patch: text_feats = cat([text_feats, prompt]), mask extended by P."""
    m = mllm if mllm is not None else pipe.mllm
    ids, mask = tokenize(prompt_text)
    feats = _read(m(input_ids=ids, attention_mask=mask, output_hidden_states=True))
    sp = SP.to(feats.dtype).unsqueeze(0).expand(feats.shape[0], -1, -1)
    mask2 = torch.cat([mask, torch.ones(mask.shape[0], sp.shape[1],
                                        dtype=mask.dtype, device=mask.device)], dim=1)
    feats = torch.cat([feats, sp], dim=1)
    assert feats.shape[1] == ids.shape[1] + sp.shape[1], (feats.shape, ids.shape, sp.shape)
    # the transformer takes the first attention_mask.sum() rows of this tensor and silently drops
    # the rest, so a short mask would lose the tail without any error
    assert mask2.shape[1] == feats.shape[1], (mask2.shape, feats.shape)
    return feats.to(dtype=pipe.mllm.dtype), mask2

# The append is correct here: generation is batch-of-1 and unpadded, so the mask stays a
# contiguous prefix. Under the batched training collator it does not — see the defect note in
# soft_prompt_ii_train_p668. The prompt being loaded was trained under that defect.
INJECTION_NOTE = "variant (ii): learnable rows appended to the MLLM output, mask extended by P"

@torch.no_grad()
def generate_with(instruction, input_images=None, negative=NEG,
                  seed=0, steps=50, text_g=5.0, image_g=2.0, size=512):
    """Sampler settings identical to every other arm. The learned prompt enters through the
    POSITIVE branch only: encode_prompt skips _get_qwen2_prompt_embeds when prompt_embeds is
    supplied and encodes the negative prompt itself, so model_pred_ref and model_pred_uncond stay
    on the stock negative embeds. `prompt` is still passed because encode_prompt chat-templates it
    before reaching the prompt_embeds branch; the templated string is discarded."""
    pe, pm = encode_injected(instruction)
    gen = torch.Generator(device=DEV).manual_seed(seed)
    res = pipe(prompt=instruction,
               prompt_embeds=pe, prompt_attention_mask=pm,
               negative_prompt=negative,
               input_images=input_images,
               width=size, height=size, num_inference_steps=steps, max_sequence_length=MAX_SEQ,
               text_guidance_scale=text_g, image_guidance_scale=image_g, cfg_range=(0.0, 1.0),
               num_images_per_prompt=1, generator=gen, output_type="pil", align_res=False)
    return res.images[0]

## 7. Equivalence gate
Nine checks, before 2.6 h of generation: the two tokenizers agree on every row of the split; the stock path reproduces the pipeline's own encoder; the wrapper agrees with the inner model, both with and without injection; the injection is a no-op once the prompt is removed; the prompt does change the conditioning; shapes and mask lengths are what the variant requires; the rope table covers the longest row; and the pipeline MLLM is numerically the same encoder train.py fitted the prompt against, which nothing else here would catch.

In [ ]:
# Gate: prove the inference-side injection reproduces the training-side one, before 2.6 h of
# generation. Two things differ between the two sides and neither was exercised by the probes,
# which contain no inference code at all:
#   - train.py encoded text with transformers Qwen2_5_VLModel and read .last_hidden_state; the
#     pipeline holds a Qwen2_5_VLForConditionalGeneration and reads .hidden_states[-1].
#   - train.py tokenized with Qwen/Qwen2.5-VL-3B-Instruct; the pipeline uses its own processor
#     tokenizer. The chat template text is identical in both, but the token ids need not be.
import torch, transformers, gc
from transformers import AutoTokenizer
print("transformers", transformers.__version__)
# requirements.txt at this commit pins 4.51.3, and soft_prompt_iii_probe recorded 4.51.3 when the
# prefix mechanism was validated. The encoder forward and, for variant (iii), DynamicCache
# semantics are version-specific, and a drift would move the conditioning silently.
assert transformers.__version__ == "4.51.3", (
    f"transformers {transformers.__version__}; this arm was trained and probed on 4.51.3")

SP = torch.load(ckpt_path(STEPS[0]), map_location="cpu")[KEY].to(DEV)
assert SP.ndim == 2 and SP.shape[0] == P, SP.shape

txt = rows[0]["instruction"]
ids, mask = tokenize(txt)

INNER = getattr(pipe.mllm, "model", None)
assert INNER is not None, ("inner model not at pipe.mllm.model; available: "
                           + str([a for a in dir(pipe.mllm) if a in ("model", "language_model", "base_model")]))
print(f"wrapper {type(pipe.mllm).__name__} | inner {type(INNER).__name__}")

print()
print("=== 1. tokenizer parity over the whole split: pipeline processor vs the training tokenizer ===")
# every row, not just one: a single row cannot show that the two tokenizers agree on the split
SYS = "You are a helpful assistant that generates high-quality images based on user instructions."
_tt = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
_bad_tpl, _bad_ids = [], []
for _j, _r in enumerate(rows):
    _t1 = _tt.apply_chat_template([{"role": "system", "content": SYS},
                                   {"role": "user", "content": _r["instruction"]}],
                                  tokenize=False, add_generation_prompt=False)
    _t2 = pipe._apply_chat_template(_r["instruction"])
    if _t1 != _t2:
        _bad_tpl.append(_j)
        continue
    _x = _tt(_t1, return_tensors="pt").input_ids
    _y = TOK(_t2, return_tensors="pt").input_ids
    if _x.shape != _y.shape or not bool((_x == _y).all()):
        _bad_ids.append(_j)
print(f"  chat template identical on {len(rows) - len(_bad_tpl)}/{len(rows)} rows")
print(f"  token ids identical on      {len(rows) - len(_bad_tpl) - len(_bad_ids)}/{len(rows)} rows")
assert not _bad_tpl, f"chat templates differ between training and inference on rows {_bad_tpl[:5]}"
assert not _bad_ids, f"the two tokenizers disagree on rows {_bad_ids[:5]}"

print()
print("=== 2. stock path reproduces the pipeline's own encoder ===")
_pe, _pm = pipe._get_qwen2_prompt_embeds(prompt=[pipe._apply_chat_template(txt)],
                                         device=DEV, max_sequence_length=MAX_SEQ)
_me, _mm = encode_stock(txt)
d = (_pe.float() - _me.float()).abs().max().item()
print(f"  max |mine - _get_qwen2_prompt_embeds| = {d:.3e}  shapes {tuple(_me.shape)} vs {tuple(_pe.shape)}")
assert _me.shape == _pe.shape and _mm.shape == _pm.shape, (_me.shape, _pe.shape)
assert d <= 1e-3, "the stock reimplementation diverges from the pipeline's own text path"

print()
print("=== 3. stock path: wrapper vs inner model ===")
s_w, sm = encode_stock(txt)
s_i, _ = encode_stock(txt, mllm=INNER)
d = (s_w.float() - s_i.float()).abs().max().item()
print(f"  max |wrapper - inner| = {d:.3e}")
assert d <= 1e-3, "wrapper and inner disagree on the stock path"

print()
print("=== 4. injected path: wrapper vs inner model (the training-vs-inference claim) ===")
f_w, fm = encode_injected(txt)
f_i, _ = encode_injected(txt, mllm=INNER)
d = (f_w.float() - f_i.float()).abs().max().item()
print(f"  max |wrapper - inner| = {d:.3e}")
assert d <= 1e-3, "the wrapper mishandles the injection; the prompt was fitted to the inner model"

print("\n=== 5. prompt removed: injected path must equal the stock path ===")
_sp_save = SP
globals()["SP"] = SP[:0]
a, am = encode_injected(txt)
globals()["SP"] = _sp_save
d = (a.float() - s_w.float()).abs().max().item()
print(f"  max |injected(P=0) - stock| = {d:.3e}")
assert a.shape == s_w.shape and am.shape == sm.shape, (a.shape, s_w.shape, am.shape, sm.shape)
assert d <= 1e-3, "the injection code perturbs the stock path even with no prompt"

print()
print("=== 6. effect: the prompt must change the conditioning ===")
n_txt = s_w.shape[1]
same = (f_w[:, :n_txt].float() - s_w.float()).abs().max().item()
tail = (f_w[:, n_txt:].float() - SP.float().unsqueeze(0)).abs().max().item()
print(f"  text rows unchanged by construction: max delta {same:.3e}")
print(f"  appended rows equal the checkpoint:   max delta {tail:.3e}  | ||SP|| = {SP.float().norm().item():.3f}")
assert f_w.shape[1] == n_txt + P, (f_w.shape, n_txt)
assert same <= 1e-3, "variant (ii) must not alter the MLLM output rows"
assert tail <= 1e-2, "the appended rows are not the loaded prompt"
assert SP.float().norm().item() > 0, "the loaded prompt is all zeros"

print()
print("=== 7. shapes and mask, three rows ===")
for _r in rows[:3]:
    _i, _ = tokenize(_r["instruction"])
    _f, _m = encode_injected(_r["instruction"])
    print(f"  instruction rows {_i.shape[1]:>4} | conditioning rows {_f.shape[1]:>4} | mask {_m.shape[1]:>4}")
    assert _m.shape[1] == _f.shape[1], (_m.shape, _f.shape)
    assert _f.shape[1] == _i.shape[1] + P, (_f.shape, _i.shape)

print()
print("=== 8. rope and sequence-length headroom over the whole split ===")
# measured WITHOUT truncation: a truncated row would report exactly MAX_SEQ and pass a <= test
lens = [TOK(pipe._apply_chat_template(r["instruction"]), return_tensors="pt").input_ids.shape[1]
        for r in rows]
cap = max(lens) + P
print(f"  templated instruction (untruncated): mean {sum(lens)/len(lens):.0f}, max {max(lens)}")
print(f"  max cap_seq_len = {cap} | released axis {_axes_before[0]} | extended axis {pipe.transformer.config.axes_lens[0]}")
assert cap <= pipe.transformer.config.axes_lens[0], "text axis still overflows after extension"
assert max(lens) < MAX_SEQ, "an instruction reaches max_sequence_length and would be truncated"
print(f"  max_sequence_length headroom: {MAX_SEQ - max(lens)} tokens, 0 rows truncated")

print()
print("=== 9. encoder checkpoint parity: pipeline MLLM vs the encoder training fitted the prompt to ===")
# The one remaining way these images could be silently wrong. train.py builds text_encoder from
# options/ft_lora.yml's pretrained_text_encoder_model_name_or_path = "Qwen/Qwen2.5-VL-3B-Instruct",
# while the pipeline's mllm is loaded from the OmniGen2/OmniGen2 repo, whose MLLM carries a trained
# "<|img|>" token on top of Qwen. Same architecture, different checkpoints. If any weight the
# instruction touches differs, the prompt is deployed against an encoder it was never fitted to and
# every image is quietly degraded while every check above still passes. Tokenizer parity does not
# cover this: an added token leaves the ids for ordinary text unchanged.
from transformers import Qwen2_5_VLModel
_train_enc = Qwen2_5_VLModel.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype=dtype).to(DEV).eval()
_e_pipe = pipe.mllm.get_input_embeddings()(ids)
_e_train = _train_enc.get_input_embeddings()(ids)
d_emb = (_e_pipe.float() - _e_train.float()).abs().max().item()
_t_stock, _ = encode_stock(txt, mllm=_train_enc)
d_stock = (_t_stock.float() - s_w.float()).abs().max().item()
_t_inj, _ = encode_injected(txt, mllm=_train_enc)
d_inj = (_t_inj.float() - f_w.float()).abs().max().item()
print(f"  vocab size: pipeline {pipe.mllm.config.vocab_size} | training {_train_enc.config.vocab_size}")
print(f"  max |input-embedding| difference on these ids = {d_emb:.3e}")
print(f"  max |stock conditioning| difference           = {d_stock:.3e}")
print(f"  max |injected conditioning| difference        = {d_inj:.3e}")
del _train_enc, _t_stock, _t_inj, _e_pipe, _e_train
gc.collect(); torch.cuda.empty_cache()
assert max(d_emb, d_stock, d_inj) <= 1e-3, (
    "the pipeline MLLM and the training MLLM produce different conditioning: the prompt was fitted "
    "to one checkpoint and would be deployed on the other. Do not generate.")
print("  the two checkpoints agree on this input: the prompt runs on what it was fitted to")

print()
print("[PASS] tokenizer, encoder checkpoint, encoder class and injection all match the training path.")

transformers 4.51.3
wrapper Qwen2_5_VLForConditionalGeneration | inner Qwen2_5_VLModel

=== 1. tokenizer parity over the whole split: pipeline processor vs the training tokenizer ===


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  vocab size: pipeline 151936 | training 151936
  max |input-embedding| difference on these ids = 0.000e+00
  max |stock conditioning| difference           = 0.000e+00
  max |injected conditioning| difference        = 0.000e+00
  the two checkpoints agree on this input: the prompt runs on what it was fitted to

[PASS] tokenizer, encoder checkpoint, encoder class and injection all match the training path.


## 8. Generate, four checkpoints (resumable)

In [ ]:
import time, gc

def gen_step(step):
    global SP
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    os.makedirs(GEN, exist_ok=True)
    have = {int(f[:-4]) for f in os.listdir(GEN) if f.endswith(".png") and f[:-4].isdigit()}
    todo = [i for i in range(len(rows)) if i not in have]
    print(f"\n=== step {step}: {len(have)} done, {len(todo)} to generate -> {GEN} ===")
    if not todo:
        return
    SP = torch.load(ckpt_path(step), map_location="cpu")[KEY].to(DEV)
    assert SP.ndim == 2 and SP.shape[0] == P, SP.shape
    t0 = time.time()
    for k, i in enumerate(todo):
        r = rows[i]
        refs = load_inputs([os.path.join(IMGS, p) for p in r["input_images"]])
        img = generate_with(r["instruction"], input_images=refs or None)
        assert img.size == (512, 512), (i, img.size)
        img.save(f"{GEN}/{i:05d}.png")
        if (k + 1) % 25 == 0:
            print(f"  {k+1}/{len(todo)}  ({(time.time()-t0)/(k+1):.1f}s/img)")
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")])
    print(f"  step {step}: {n}/{len(rows)} images")
    assert n == len(rows), "incomplete"

for step in STEPS:
    gen_step(step)
gc.collect(); torch.cuda.empty_cache()
print("\nall steps generated")


=== step 500: 0 done, 98 to generate -> /content/drive/MyDrive/Colab Notebooks/My Master's Thesis/Data/gen/val/sp_ii_p668__step500 ===


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

...

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  step 3000: 98/98 images

all steps generated


## 9. Manifests

In [ ]:
import json, datetime, torch

for step in STEPS:
    GEN = f"{DRIVE}/gen/{DATASET}/{RUN}__step{step}"
    n = len([f for f in os.listdir(GEN) if f.endswith(".png")]) if os.path.isdir(GEN) else 0
    if n == 0:
        continue
    man = {"dataset": DATASET, "run": RUN, "variant": VARIANT, "step": step, "P": P, "n_images": n,
           "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__,
           "prompt_checkpoint": ckpt_path(step),
           "injection": INJECTION_NOTE,
           "cfg_branch": "positive only: prompt_embeds passed precomputed, negative branch encoded by the pipeline",
           "rope_axes_lens": list(pipe.transformer.config.axes_lens),
           "sampler": {"steps": 50, "text_guidance_scale": 5.0, "image_guidance_scale": 2.0,
                       "cfg_range": [0.0, 1.0], "max_sequence_length": MAX_SEQ, "seed": 0,
                       "size": 512, "align_res": False},
           "negative_prompt": NEG,
           "written": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(man, open(f"{GEN}/manifest.json", "w"), indent=2)
print("manifests written")

manifests written
